# Run the Nova Sales Assistant on Colab

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. Open the `trycloudflare.com` link that step 3 prints. It works while this notebook is running.

The first start downloads Qwen2.5-7B-Instruct (about 15 GB) and takes 5 to 10 minutes. You can run every cell again safely.

### 1. Check the GPU and get the code

In [ ]:
import os, subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU. Choose Runtime > Change runtime type > T4 GPU, then run again.")
print("GPU:", gpu.stdout.strip())

os.chdir("/content")
if os.path.isdir("nova-sales-assistant"):
    !git -C nova-sales-assistant pull -q
else:
    !git clone -q https://github.com/Hagar202/nova-sales-assistant.git
os.chdir("/content/nova-sales-assistant")
!pip install -q -r requirements.txt
print("Code and packages are ready.")

### 2. Start the API server
The model is loaded once. This cell waits until the server answers.

In [ ]:
import time, urllib.request

subprocess.run(["pkill", "-f", "uvicorn app.main:app"])  # stop an old server if you run this again
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)

started = time.time()
while True:
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=5)
        print(f"Server is ready after {int(time.time() - started)} s")
        break
    except Exception:
        if server.poll() is not None:
            print(open("server.log").read()[-3000:])
            raise RuntimeError("The server stopped. See the log above.")
        print(f"Loading the model... {int(time.time() - started)} s", end="\r")
        time.sleep(10)

### 3. Open a public link
Cloudflare Tunnel is free and needs no account.

In [ ]:
import re

if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

subprocess.run(["pkill", "-f", "cloudflared tunnel"])
# The tunnel writes to a file, not a pipe, so it never blocks when nobody reads its output
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if match:
        url = match.group()
        break
    time.sleep(1)

if url:
    print("Web app :", url)
    print("API docs:", url + "/docs")
    print("\nThe link can take 10 to 20 seconds to start working.")
else:
    print(open("tunnel.log").read()[-2000:])
    print("No link yet. Run this cell again.")

### Server log (for debugging)

In [ ]:
!tail -n 30 server.log